# SpMV barrier re-measure (2026-10-07)

Barrier-verified results for the streaming SpMV: 65,536 PEs (256×256), 114,000² at d=1e-3 (13.0 M non-zeros),
header vs valtask encoding, full vs column H2D mode, 1 / 8 / 16 I/O channels. Data:
`2026-10-07_barrier-remeasure.csv` and `2026-10-07_s1-column-matrix.csv` under `lib.DATA`. Only barrier-closed times
(`stream_time_corrected`, `setup_h2d_time_corrected`) are used for conclusions; host-return times appear only to
show their gap. These CSVs have no `system` column, so a bad system cannot be filtered by name; the one known bad-system run
never produced a row. Compare within a session only (session drift, `docs/results.md`).

Hypotheses were written down per question before the tables were looked at *for this notebook*, but the team already
knew the headline numbers (`docs/results.md`); the predictions are those recorded in `docs/val-only-encoding.md` and
`docs/streaming-rewrite-status.md`, and where they were wrong the text says so.

In [ ]:
import sys, pathlib
sys.path.insert(0, str(pathlib.Path.cwd()))
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display, Markdown
import lib
lib.setup_plots()
pd.set_option("display.width", 200, "display.max_columns", 30)

---
## 1. Timing basis: how far below the truth are host-return times?

### Hypothesis

**Claim under test** (docs/host-timing-barrier.md): `memcpy_h2d_time` (host return) is a lower
bound on the true streaming time, because `wait_ew(nonblock=False)` returns before the device is done and the
next blocking call absorbs the rest. `stream_time_corrected` (barrier-closed) is the valid time.

**Prediction.** Corrected ≥ host-return on every row. The gap (the *tail*) is largest where the device is
slowest to drain: streaming at 1 channel (the 10-07 log showed a 97 ms host return against 535 ms corrected),
and smallest in column mode, where the barrier re-measure found host-return and corrected nearly equal.
**Falsifier:** corrected < host-return on any row would mean the barrier is not a valid completion signal.

### Result

In [ ]:
raw = lib.load_spmv()
df, ex = lib.valid_spmv(raw)
print("timing basis: stream_time_corrected (barrier-closed) vs memcpy_h2d_time (host-return, lower bound)")
print(f"{len(raw)} rows loaded from {raw['source'].nunique() if len(raw) else 0} file(s) |", ex, f"| usable: {len(df)}")
if df.empty:
    lib.no_data("2026-10-07_barrier-remeasure.csv")
else:
    df["tail_ms"] = (df.stream_time_corrected - df.memcpy_h2d_time) * 1e3
    df["tail_frac"] = df.tail_ms / (df.stream_time_corrected * 1e3)
    print("rows where corrected < host-return:", int((df.tail_ms < -0.05).sum()), "(tolerance 0.05 ms)")
    t = df.groupby(["encoding", "h2d_mode", "channels"]).agg(
        n=("tail_ms", "size"), host_ms_med=("memcpy_h2d_time", lambda s: s.median() * 1e3),
        corr_ms_med=("stream_time_corrected", lambda s: s.median() * 1e3),
        tail_ms_min=("tail_ms", "min"), tail_ms_med=("tail_ms", "median"), tail_ms_max=("tail_ms", "max"),
        tail_frac_med=("tail_frac", "median")).round(2)
    display(t)
    fig, axs = lib.panels(2, sharey=True, sharex=True)
    for ax, hm in zip(axs, ["full", "column"]):
        for enc, g in df[df.h2d_mode == hm].groupby("encoding"):
            st = lib.style("encoding", enc)
            ax.scatter(g.memcpy_h2d_time * 1e3, g.stream_time_corrected * 1e3, s=40, color=st["color"], marker=st["marker"],
                       alpha=0.9, edgecolors="white", linewidths=0.6, label=enc, zorder=3)
        ax.plot([5, 700], [5, 700], color=lib.INK, lw=1.2, ls=(0, (1, 3)), label="corrected = host-return", zorder=2)
        lib.finish(ax, "host-return time, a lower bound (ms)", "barrier-closed time (ms)", f"{hm} mode, 65,536 PEs",
                   legend="encoding", logx=True, logy=True, xticks=[10, 30, 100, 300], yticks=[10, 30, 100, 300, 1000])
    plt.show()

### What that means

- The barrier-closed time is never below the host-return time (0 of 28 rows), so the check that could have broken the
  barrier's premise did not fire.
- In **full** mode the host returns long before the device is done: the tail is 82% of the time for header at 1 channel (438 ms of 535), 94% for
  valtask/val at 1 channel, 41% / 33% at 8 and ~13% at 16 channels. In **column** mode at 8 and 16 channels the two agree to ≤0.1 ms: nothing is
  left buffered when the call returns. At 1 channel column mode still has a tail (65 ms of 107, n=1).
- **Not established:** that the barrier is *exact*. It bounds the true time from above by at most its own
  cost (0.2 ms idle); an independent on-wafer tsc check is still held.

### What the vendor says

**Agrees.** [BW] lines 57–58: "To record the elapsed time on host may not show the true timing because the runtime
may not start the transaction when the user calls a H2D/D2H command." The vendor's own benchmark therefore times
on the device ("Instead, this bandwidhTest samples the timing on the device.", [BW]:60).

**Agrees.** [SDK]:257 "If blocking mode (`nonblock=False`) is selected, the API waits until the operation is done."
and, for nonblocking mode, "the function returns before the operation even starts" ([VCM] §4 quotes the full
sentence). Our finding is the narrower point that "done" for a *launched function* is not "device finished".

Sources: **[SDK]** https://sdk.cerebras.ai/tensor-streaming ·
**[BW]** `csl-examples/benchmarks/bandwidth-test/run.appliance.py` (csl-examples
targets SDK 1.4.0; the cluster runs 2.10.0) · **[CLI]** `--channels` / `--width-*-buf` help,
cerebras-sdk 2.10.0 · **[VCM]** `docs/vendor-claims-vs-measurements.md`.
[SDK]:N cites a line of `analysis/vendor_sources/2026-10-08_tensor-streaming.md`, a snapshot of
https://sdk.cerebras.ai/tensor-streaming.md taken 2026-10-08; every [SDK] quote was checked against it with `grep -F`.

---
## 2. Header vs valtask (full mode, row-major): does dropping the header words pay?

### Hypothesis

**Reasoning** (docs/val-only-encoding.md): header streams 29.9 M words (119.5 MB) per pass and valtask
16.97 M (67.9 MB), 1.76× fewer; valtask also removes per-word decode work. If the transfer is **link-bound** the
speed-up should track the byte ratio, ≈1.76×. If it is **consumption-bound** (task cost per word) it could be larger.

**Prediction:** valtask faster than header at every channel count, by ≥1.76× at 1 channel.
**Bulk `val` (one `@fmovs`)** was predicted to equal valtask on bytes but removes the per-word task, hence faster;
the concurrency hang seen in hardware is a separate matter.
**Falsifier:** a speed-up well below 1.76× at 1 channel, or valtask slower than header.

### Result

In [ ]:
if not df.empty:
    f = df[(df.h2d_mode == "full") & (df.memcpy_order == "row") & (df.setup_order == "row")]
    t = lib.summarize(f, ["encoding", "channels"], "stream_time_corrected", 1e3, 1)
    t = t.rename(columns={"min": "min ms", "median": "median ms", "max": "max ms"})
    display(t)
    piv = t.pivot(index="channels", columns="encoding", values="median ms")
    if {"header", "valtask"} <= set(piv.columns):
        piv["header/valtask (median)"] = (piv["header"] / piv["valtask"]).round(2)
    display(piv)
    fig, axs = lib.panels(1)
    lib.strip_plot(axs[0], f.assign(ms=f.stream_time_corrected * 1e3), "channels", "ms", "encoding", "encoding")
    lib.finish(axs[0], "I/O channels (count)", "barrier-closed stream time (ms)", "full / row-major: every run, line = median",
               legend="encoding", logx=True, logy=True, xticks=[1, 8, 16], yticks=[30, 100, 300, 1000])
    plt.show()

### What that means

- **valtask beats header at every channel count on the median** (2.6× at 1 channel, 1.9× at 8, 1.8× at 16 on medians of 180 vs 101 ms;
  min against min at 16 channels is only 1.1×, 85.9 vs 76.7 ms, and the ranges 86–268 and 77–123 ms overlap). Row-major setup only,
  so the rows are one session's. At 1 channel 2.6× exceeds the 1.76× byte
  ratio, so header was *not* purely link-bound there: per-word work matters too.
- At 16 channels the spread (3× between identical header runs) is as large as the encoding effect: n=3 per side is a hint, not a ranking.
- Bulk `val` is measured at 1 channel only (194 ms, ~5% under valtask) because it hangs when streams overlap.
- **Not established:** a 16-channel ranking. n=3 with 3× noise is not enough; that is what S2 (n≥3, alternating) is for.

### What the vendor says

**Silent.** No vendor text compares stream encodings. Their closest code is a scalar loop for the indexed
accumulate, which docs/val-only-encoding.md ("Kernel decision") says `benchmarks/spmv-hypersparse` also does as a scalar loop
(not re-checked here; the kernel is under `benchmarks/spmv-hypersparse/src/hypersparse_spmv/pe.csl`). [BW]:44 on aggregation: "runtime can aggregate multiple nonblocking H2D/D2H commands implicitly."
Nothing in the vendor material says fewer words per nonzero is faster; our 2.6× is our own result.

Sources: **[SDK]** https://sdk.cerebras.ai/tensor-streaming ·
**[BW]** `csl-examples/benchmarks/bandwidth-test/run.appliance.py` (csl-examples
targets SDK 1.4.0; the cluster runs 2.10.0) · **[CLI]** `--channels` / `--width-*-buf` help,
cerebras-sdk 2.10.0 · **[VCM]** `docs/vendor-claims-vs-measurements.md`.
[SDK]:N cites a line of `analysis/vendor_sources/2026-10-08_tensor-streaming.md`, a snapshot of
https://sdk.cerebras.ai/tensor-streaming.md taken 2026-10-08; every [SDK] quote was checked against it with `grep -F`.

---
## 3. Do more I/O channels speed up streaming?

### Hypothesis

**Prediction:** yes. Copy-mode setup scaled 1→16 channels (0.33 → 5.2 GB/s row-major) and nothing in streaming
should cap that unless the PE-side task is the bottleneck. The 2026-10-03 conclusion ("consumption-bound,
channel-flat") rested on host-return times and was predicted to disappear under the barrier.
**Falsifier:** corrected times flat between 8 and 16 channels with spread smaller than the effect.

### Result

In [ ]:
if not df.empty:
    f = df[(df.h2d_mode == "full") & (df.memcpy_order == "row") & (df.setup_order == "row") & df.encoding.isin(["header", "valtask"])]
    t = lib.summarize(f, ["encoding", "channels"], "stream_time_corrected", 1e3, 1)
    base = t[t.channels == 1].set_index("encoding")["median"]
    t["speed-up vs 1 ch (median)"] = [round(base[e] / m, 2) for e, m in zip(t.encoding, t["median"])]
    t["GB/s (median, 1 pass)"] = [round(f[(f.encoding == e) & (f.channels == c)].h2d_bytes_total.iloc[0] / (m * 1e-3) / 1e9, 2)
                                  for e, c, m in zip(t.encoding, t.channels, t["median"])]
    display(t.rename(columns={"min": "min ms", "median": "median ms", "max": "max ms"}))

### What that means

- Streaming scales with channels (row-major setup, full mode): header 535 → 152 → 180 ms (16 ch: n=3, min 86), valtask 205 → 81 → 101 ms
  (medians; n=1 at 1 and 8 channels) for 1 / 8 / 16 channels. The 1→16 speed-up is 6.2× on the best header run (3.0× on the median) and 2.0× on the valtask median.
- 8 → 16 channels is not reliably faster for valtask (81 ms vs a 77–123 ms range): the flat-looking top end is
  noise-dominated, not proven flat.
- The "channel-flat" reading of 10-03 is withdrawn (`docs/results.md`). **Not established:** where scaling stops.
  Streaming is at 0.3–1.6 GB/s here against 5–7 GB/s in copy mode, so some other limit remains (see M2).

### What the vendor says

**Agrees with direction.** [BW]:25: "The more channels, the higher bandwidth". [BW]:18–19: "The maximum bandwidth of a
single channel is around 7Gbps (Giga bit per second)." (= 0.875 GB/s). This is a comment in the vendor's example code (SDK 1.4.0 examples), not SDK documentation.

**Contradicts in magnitude.** Our verified 1-channel copy rate is 0.33 GB/s (2.6 Gbps), and streaming 0.22–0.33 GB/s,
below a third of the quoted per-channel maximum. The vendor figure may refer to different hardware or SDK (the
file targets SDK 1.4.0); we cannot tell which.

**Unsourced claim withdrawn.** "Performance improvements are minimal past 8 channels" is attributed to the docs in our
commits `24d995a` and `4040625`; no such sentence exists in [SDK] or [CLI] ([VCM] §3).

Sources: **[SDK]** https://sdk.cerebras.ai/tensor-streaming ·
**[BW]** `csl-examples/benchmarks/bandwidth-test/run.appliance.py` (csl-examples
targets SDK 1.4.0; the cluster runs 2.10.0) · **[CLI]** `--channels` / `--width-*-buf` help,
cerebras-sdk 2.10.0 · **[VCM]** `docs/vendor-claims-vs-measurements.md`.
[SDK]:N cites a line of `analysis/vendor_sources/2026-10-08_tensor-streaming.md`, a snapshot of
https://sdk.cerebras.ai/tensor-streaming.md taken 2026-10-08; every [SDK] quote was checked against it with `grep -F`.

---
## 4. Full vs column H2D mode (header kernel) and the effect of setup order

### Hypothesis

**Reasoning.** Column mode sends one 1-wide rectangle per grid column with exact per-column lengths (112.1 MB vs
119.5 MB in full mode, −6%). docs/streaming-rewrite-status.md argued a 1-wide rectangle cannot use more than one
channel, which predicts column mode **does not scale with channels** and loses to full mode at high channel counts.
Setup order col-major was predicted to help copy-mode setup (vendor claim).
**Falsifier:** column mode faster than full at 16 channels despite 1-wide rectangles.

### Result

In [ ]:
if not df.empty:
    f = df[(df.encoding == "header") & (df.memcpy_order == "row")]
    t = lib.summarize(f, ["h2d_mode", "channels", "setup_order", "serialize_column_rects"], "stream_time_corrected", 1e3, 1)
    display(t.rename(columns={"min": "min ms", "median": "median ms", "max": "max ms"}))
    fig, axs = lib.panels(1)
    lib.strip_plot(axs[0], f.assign(ms=f.stream_time_corrected * 1e3), "channels", "ms", "h2d_mode", "h2d_mode")
    lib.finish(axs[0], "I/O channels (count)", "barrier-closed stream time (ms)", "header, row-major stream order",
               legend="H2D mode", logx=True, logy=True, xticks=[1, 8, 16], yticks=[30, 100, 300, 1000])
    plt.show()
    mo = df[(df.encoding == "header") & (df.h2d_mode == "full") & (df.channels == 16)]
    print("header, full, 16 ch by memory order (stream order of the data):")
    display(lib.summarize(mo, ["memcpy_order", "setup_order"], "stream_time_corrected", 1e3, 1))
    v = df[(df.encoding == "valtask") & (df.h2d_mode == "column")]
    if len(v):
        print("valtask in column mode (S1 file, serialized rects):")
        display(lib.summarize(v, ["channels", "setup_order"], "stream_time_corrected", 1e3, 1))

### What that means

- **The prediction is false.** Column mode is faster than full at every channel count (header, medians: 107 vs 535 ms at 1 ch,
  59 vs 152 ms at 8 (n=3 vs 1), 62–66 vs 86–268 ms at 16) and far steadier at 16 channels (61.4–69.4 ms over 6 column runs, setup order and
  serialization mixed; full mode varies 3×).
  Column mode also carries only 6% fewer bytes, so the gain is not payload. Serializing the rectangles (n=1, 69.4 ms) did not help header at 16 channels.
- Column mode does *not* improve from 8 to 16 channels (59 → 62–66 ms), consistent with 1-wide rectangles not spreading
  over channels, but at 8–16 it is already faster than every full-mode run.
- Valtask in column mode (from the S1 file, rects serialized) is the fastest streaming seen: about 40–49 ms.
  Those rows come from the later S1 session; compare to the first session's rows with the drift caveat (`docs/results.md`, "Timing basis").
- Memory order for streaming (header, full, 16 ch): the one col-major run (245 ms) lies inside the row-major range (86 / 180 / 268 ms, n=3): col-major is not better, and one run cannot say more.
- **Not established:** why. M4 (strips k = 1 … W, serial vs all) isolates concurrency/granularity from layout.

### What the vendor says

**Silent on column mode for streaming.** [SDK]:313: "Column-major delivers better bandwidth than row-major."
It concerns the memory order of the data buffer, not the rectangle shape, and the vendor benchmark measures copy mode only
(`streaming=False` throughout, e.g. [BW]:344, 360, 384, 404). Our column H2D *mode* (many narrow rectangles) has no vendor counterpart.
[BW]:44: "runtime can aggregate multiple nonblocking H2D/D2H commands implicitly." is the nearest statement; it
suggests many nonblocking calls need not cost more than one.

Sources: **[SDK]** https://sdk.cerebras.ai/tensor-streaming ·
**[BW]** `csl-examples/benchmarks/bandwidth-test/run.appliance.py` (csl-examples
targets SDK 1.4.0; the cluster runs 2.10.0) · **[CLI]** `--channels` / `--width-*-buf` help,
cerebras-sdk 2.10.0 · **[VCM]** `docs/vendor-claims-vs-measurements.md`.
[SDK]:N cites a line of `analysis/vendor_sources/2026-10-08_tensor-streaming.md`, a snapshot of
https://sdk.cerebras.ai/tensor-streaming.md taken 2026-10-08; every [SDK] quote was checked against it with `grep -F`.

---
## 5. Setup copy: row-major vs col-major, by channels

### Hypothesis

**Prediction (vendor):** col-major setup is faster than row-major in copy mode. We measured +43% on 10-06 using
host-return times, which overstate rates; the barrier-closed gain is predicted to be smaller.
**Falsifier:** no difference, or col-major slower, on `setup_h2d_time_corrected`.

### Result

In [ ]:
if not df.empty:
    f = df.copy()
    f["setup_gbps"] = f.setup_h2d_bytes / f.setup_h2d_time_corrected / 1e9
    t = lib.summarize(f, ["channels", "setup_order", "source"], "setup_gbps", 1.0, 2)
    t = t.rename(columns={"min": "min GB/s", "median": "median GB/s", "max": "max GB/s"})
    print("timing basis: setup_h2d_time_corrected (barrier-closed); 302 MB copy-mode setup")
    display(t)
    f["stream_gbps"] = f.h2d_bytes_total / f.stream_time_corrected / 1e9
    fig, axs = lib.panels(3, sharey=True, sharex=True)
    lib.strip_plot(axs[0], f, "channels", "setup_gbps", "setup_order", "order", labels={"row": "row-major setup", "col": "col-major setup"}, endlabel=False)
    lib.draw_claims_vs_channels(axs[0])
    lib.finish(axs[0], "I/O channels (count)", "setup copy rate (GB/s), line = median", "setup copy, 302 MB, barrier-closed",
               legend="setup order", logx=True, logy=True, xticks=[1, 8, 16], yticks=[0.3, 1, 3, 10])
    for ax, hm in zip(axs[1:], ["full", "column"]):
        lib.strip_plot(ax, f[f.h2d_mode == hm], "channels", "stream_gbps", "encoding", "encoding", endlabel=False)
        lib.draw_claims_vs_channels(ax)
        lib.finish(ax, "I/O channels (count)", "streaming rate, one pass (GB/s), line = median", f"streaming, {hm} mode, barrier-closed",
                   legend="encoding", logx=True, logy=True, xticks=[1, 8, 16], yticks=[0.3, 1, 3, 10])
    lib.place_claims(fig); plt.show()
    display(lib.aggregate_note(max(f.setup_gbps.max(), f.stream_gbps.max())))

### What that means

- At 16 channels col-major setup runs at a median of 6.66 GB/s (4.85–7.21, n=9) against 5.12 (3.72–5.20, n=8) row-major: **about +30% on medians**, below
  the +43% from host-return times. The ranges overlap and mix sessions (see the `source` split in the table): the session-to-session spread
  of setup rate is itself ~±20%, so +30% is an estimate, not a constant.
- At 8 channels col-major (n=2, 5.36 GB/s) is far above row-major (median 3.06, n=5), a larger gain than at 16; 1 channel has row-major only.
- Setup scales with channels (0.37 → 3.06 → 5.12 GB/s row-major medians), the same shape as streaming.
- **Not established:** the col/row gain at 1 channel, and its size at 8 (n=2).

### What the vendor says

**Agrees.** [SDK]:313: "Column-major delivers better bandwidth than row-major." Our gain is +30% (barrier-verified, 16
channels, 302 MB). Mismatch of detail: [SDK] prose says for column-major "w is the fastest varying dimension" while the vendor
code ([VCM] §2, `hwl_2_oned_colmajor`, `run.appliance.py:178`) makes **h** fastest; hardware validation follows the code.
**Contradicts** the 10.35 GB/s figure from 10-06 only in that it was a host-return rate and is an overestimate.

Sources: **[SDK]** https://sdk.cerebras.ai/tensor-streaming ·
**[BW]** `csl-examples/benchmarks/bandwidth-test/run.appliance.py` (csl-examples
targets SDK 1.4.0; the cluster runs 2.10.0) · **[CLI]** `--channels` / `--width-*-buf` help,
cerebras-sdk 2.10.0 · **[VCM]** `docs/vendor-claims-vs-measurements.md`.
[SDK]:N cites a line of `analysis/vendor_sources/2026-10-08_tensor-streaming.md`, a snapshot of
https://sdk.cerebras.ai/tensor-streaming.md taken 2026-10-08; every [SDK] quote was checked against it with `grep -F`.